# Short PyTorch warm-up

We use three supplied examples to introduce tensor shapes, differentiation
through successive holdings, and parameter updates. These examples prepare
you to write the functions in `starter.ipynb`; they are not an additional
submission task. Run the code cells in order, considering the expected output
before each one.

## Tensors and network outputs

A *tensor* is a numerical array. In the first cell, `x` has shape `(1, 3)`:
one row containing three features for one path. Its entries represent the
fraction of time remaining, log-moneyness, and the previous holding.

The network maps these three inputs through a hidden layer of width eight
to one output. `nn.Linear` applies weights and biases, `nn.Tanh` applies a
nonlinear activation, and the final `nn.Sigmoid` constrains the output to
between zero and one.

The output `y` has shape `(1, 1)`. The operation `y.squeeze(-1)` removes its
final size-one dimension, giving shape `(1,)` without changing the value.
The remaining dimension still indexes paths. Run the cell and compare the
three printed shapes.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
x = torch.tensor([[0.5, 0.0, 0.0]])  # one path, three features
net = nn.Sequential(nn.Linear(3, 8), nn.Tanh(), nn.Linear(8, 1), nn.Sigmoid())
y = net(x)
print("input", x.shape, "output", y.shape, "holding vector", y.squeeze(-1).shape)

## Differentiating through previous holdings

We now consider two holdings determined by scalar parameters $\theta$ and
$\beta$:
$$\delta_0=\theta,\qquad \delta_1=\theta+\beta\delta_0.$$
With price changes $1$ and $-2$, the trading gain is
$$G=\delta_0-2\delta_1=-\theta-2\beta\theta.$$
At $\theta=1/4$ and $\beta=1/2$, the gain is $-1/2$, and its derivatives are
$\partial G/\partial\theta=-2$ and $\partial G/\partial\beta=-1/2$.
The first calculation below uses `torch.autograd.grad` to return these two
derivatives in that order; it does not update the parameters.

The second calculation passes `delta0.detach()` to the expression for
`delta1`. This retains the value of the first holding but treats that input
as constant during differentiation. The gain stays the same, while the
reported derivative with respect to $\theta$ becomes $-1$. It omits the effect
of $\theta$ on the second holding through the first. Compare the two printed
results to see why the assignment's rollout must retain this dependence.

In [ ]:
theta = torch.tensor(0.25, requires_grad=True)
beta = torch.tensor(0.5, requires_grad=True)
delta0 = theta
delta1 = theta + beta*delta0
gain = delta0 - 2*delta1
print("gain", gain.item(), "derivatives", torch.autograd.grad(gain, (theta, beta)))

# Same forward gain, but a different computation graph:
theta = torch.tensor(0.25, requires_grad=True)
beta = torch.tensor(0.5, requires_grad=True)
delta0 = theta
delta1 = theta + beta*delta0.detach()
gain = delta0 - 2*delta1
print("detached gain", gain.item(), "derivatives", torch.autograd.grad(gain, (theta, beta)))

## Updating a parameter

We next minimize $(a-2)^2$, starting from $a=0$. The statement
`nn.Parameter(...)` marks `a` as a learnable parameter, and the optimizer
updates it by gradient descent with learning rate $0.1$.

Each iteration clears the gradients with `zero_grad`, calculates the
objective, differentiates it with `backward`, and updates the parameter with
`step`. Clearing the gradients is necessary here because successive
`backward` calls otherwise add to the stored gradients. Run the cell and
observe how `a` moves toward the minimizer $2$.

The assignment uses this same sequence with a different optimizer and
objective. Its optimizer updates the network parameters together with the
single expected-shortfall threshold $\eta$.

In [ ]:
a = nn.Parameter(torch.tensor(0.0))
opt = torch.optim.SGD([a], lr=0.1)
for step in range(5):
    opt.zero_grad(set_to_none=True)
    objective = (a-2).square()
    objective.backward()
    opt.step()
    print(step+1, "parameter", round(a.item(), 4))

A parameter update uses the derivative of the objective at the current
parameter values. In the hedging assignment, that objective depends on the
terminal loss, which we calculate after simulating the whole path. Each
holding still uses only the information available at its decision date.
Using the terminal loss to train the parameters therefore preserves the
information restriction on the strategy.

Continue with the first code cell in `starter.ipynb`, which imports the
supplied functions and creates the configuration. Then choose one experiment
and write the three TODO functions. The separate course beginner notebook
provides optional further examples of tensors, networks, validation, and
gradient checks.